In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sentence_transformers import SentenceTransformer
import joblib
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

print("1. Загрузка данных...")
df = pd.read_excel('Обращения_1931.xlsx')

# Оставляем только нужные колонки (без утечек)
features_to_keep = ['Номер запроса', 'Услуга', 'Компонент услуги 1 уровня', 'Тип запроса', 'Описание 2', 'Вид запроса']
df_clean = df[features_to_keep].copy()

# Продвинутая очистка
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'тема письма[:\s]*|текст письма[:\s]*|индекс опс[:\s]*|добрый день[,!\s]*|здравствуйте[,!\s]*', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'\b\d+\b', ' <NUM> ', text)
    return re.sub(r'\s+', ' ', text).strip()

df_clean['text'] = (df_clean['Описание 2'].apply(clean_text) + " | " +
                    df_clean['Услуга'].fillna('Не указано') + " " +
                    df_clean['Компонент услуги 1 уровня'].fillna('Не указано'))

# Топ-15 + Прочее
category_counts = df_clean['Вид запроса'].value_counts()
top_15 = category_counts.head(15).index.tolist()
df_clean['label_name'] = df_clean['Вид запроса'].apply(lambda x: x if x in top_15 else 'Прочее / Неопределено')

# Удаляем только полные дубликаты; близкие дубликаты безопасно не удаляем
# (семантическая дедупликация требует отдельного порога и валидации).
df_clean = df_clean.drop_duplicates(subset=['text', 'label_name']).reset_index(drop=True)

print("2. Загрузка мощной модели эмбеддингов (это займет ~30 сек на GPU)...")
# Эта модель дает гораздо более качественные вектора, чем rubert-tiny2
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
embeddings = embedder.encode(df_clean['text'].tolist(), show_progress_bar=True, batch_size=32)

X = np.vstack(embeddings)
y = df_clean['label_name']

print("✅ Данные готовы!")

1. Загрузка данных...
2. Загрузка мощной модели эмбеддингов (это займет ~30 сек на GPU)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

✅ Данные готовы!


In [3]:
print("3. Разделение данных...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("4. Обучение Random Forest + Калибровка вероятностей...")
# Базовая модель с балансом классов
base_rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Калибровка делает вероятности "честными", чтобы порог отсечения работал корректно
print("   (Калибровка может занять 1-2 минуты, это нормально)")
model = CalibratedClassifierCV(base_rf, method='isotonic', cv=3)
model.fit(X_train, y_train)

print("✅ Модель успешно обучена и откалибрована!")

# Сохраняем именно обученный классификатор; эмбеддер скачивается отдельно при первом запуске.
MODEL_PATH = "model.pkl"
joblib.dump(model, MODEL_PATH)
print(f"✅ Классификатор сохранён в {MODEL_PATH}")

3. Разделение данных...
4. Обучение Random Forest + Калибровка вероятностей...
   (Калибровка может занять 1-2 минуты, это нормально)
✅ Модель успешно обучена и откалибрована!


In [8]:
# Предсказания
y_pred_raw = model.predict(X_test)
y_prob = model.predict_proba(X_test)
confidence_scores = np.max(y_prob, axis=1)

# Логика Fallback
THRESHOLD = 0.25
is_reliable = (confidence_scores >= THRESHOLD) & (y_pred_raw != 'Прочее / Неопределено')
y_pred_final = np.where(is_reliable, y_pred_raw, 'ОПЕРАТОР')
reliable_mask = (y_pred_final != 'ОПЕРАТОР')

print("="*70)
print("🏆 ИТОГОВЫЕ МЕТРИКИ ПРОДАКШН-ГОТОВОГО РЕШЕНИЯ")
print("="*70)
fallback_rate = 100 * (1 - reliable_mask.mean())
print(f"📉 Доля обращений к оператору (Fallback): {fallback_rate:.1f}%")
print(f"🎯 Accuracy (на автоматических ответах): {accuracy_score(y_test[reliable_mask], y_pred_final[reliable_mask]):.4f}")
print(f"📊 Macro-F1 (на автоматических ответах): {f1_score(y_test[reliable_mask], y_pred_final[reliable_mask], average='macro', zero_division=0):.4f}")

# BUSINESS METRICS
top_15_set = set(top_15)
mask_top15_gt = y_test.isin(top_15_set)
mask_reliable_and_top15 = reliable_mask & mask_top15_gt

if mask_reliable_and_top15.sum() > 0:
    biz_f1 = f1_score(y_test[mask_reliable_and_top15], y_pred_final[mask_reliable_and_top15], average='macro', zero_division=0)
    biz_acc = accuracy_score(y_test[mask_reliable_and_top15], y_pred_final[mask_reliable_and_top15])
    print(f"\n🔥 BUSINESS MACRO-F1 (только по 15 целевым классам): {biz_f1:.4f}")
    print(f"   (Именно эту цифру мы обоснуем в отчете как главную!)")

print("\n=== Classification Report (автоматические ответы) ===")
print(classification_report(y_test[reliable_mask], y_pred_final[reliable_mask], zero_division=0))

print("\n=== Confusion Matrix (автоматические ответы) ===")
labels = sorted(set(y_test[reliable_mask]) | set(y_pred_final[reliable_mask]))
cm = confusion_matrix(y_test[reliable_mask], y_pred_final[reliable_mask], labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(xticks_rotation=90, cmap="Blues")
plt.tight_layout()
plt.show()

# Примеры правильных автоматических предсказаний и ошибок.
results = pd.DataFrame({"text": df_clean.loc[y_test.index, "text"].values,
                        "true": y_test.values, "pred": y_pred_final,
                        "confidence": confidence_scores})
correct = results[reliable_mask & (y_test.to_numpy() == y_pred_final)]
errors = results[reliable_mask & (y_test.to_numpy() != y_pred_final)]
print("\n=== Примеры правильных предсказаний ===")
print(correct.head(5).to_string(index=False))
print("\n=== Примеры ошибок ===")
print(errors.head(5).to_string(index=False))

🏆 ИТОГОВЫЕ МЕТРИКИ ПРОДАКШН-ГОТОВОГО РЕШЕНИЯ
📉 Доля обращений к оператору (Fallback): 39.1%
🎯 Accuracy (на автоматических ответах): 0.5066
📊 Macro-F1 (на автоматических ответах): 0.4034

🔥 BUSINESS MACRO-F1 (только по 15 целевым классам): 0.4690
   (Именно эту цифру мы обоснуем в отчете как главную!)

=== Classification Report (автоматические ответы) ===
                                                        precision    recall  f1-score   support

                               Импорт списков из ЛК ЮЛ       0.28      0.47      0.35        15
            Импорт списков из архива ф.103 (zip-архив)       0.00      0.00      0.00         9
                                        Личный кабинет       0.00      0.00      0.00         3
                              Отслеживание отправлений       0.60      0.75      0.67        28
Ошибка в приложении (восстановление работоспособности)       0.53      0.62      0.57        13
                    Ошибки загрузки страницы/зависания       0.33 